<a href="https://colab.research.google.com/github/D-F-V/Project_brief/blob/main/exploracion_madrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Celda 1.** Instala rdata, la librería que sabe leer archivos de R. El resto (pandas, numpy) ya viene en Colab.


In [2]:
!pip install -q rdata

**Celda 2.** Importa librerías y muestra sus versiones, para dejar constancia del entorno.

In [3]:
import hashlib
import platform
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rdata

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("rdata:", rdata.__version__)

Python: 3.13.18
pandas: 2.2.3
numpy: 2.4.6
rdata: 1.1.0


**Celda 3.** Define de dónde viene el archivo y lo descarga a la carpeta `data/` de la sesión. La URL apunta al commit fijado, no a `master.`


In [4]:
REPO = "paezha/idealista18"
COMMIT = "e9ef80618f8a9cc28f672ac138f298bbd519918e"
RUTA_EN_REPO = "data/Madrid_Sale.rda"
URL = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{RUTA_EN_REPO}"

CARPETA_DATOS = Path("data")
CARPETA_DATOS.mkdir(exist_ok=True)
RUTA_LOCAL = CARPETA_DATOS / "Madrid_Sale.rda"

if RUTA_LOCAL.exists():
    print("El archivo ya existe, no se vuelve a descargar:", RUTA_LOCAL)
else:
    urllib.request.urlretrieve(URL, RUTA_LOCAL)
    print("Descargado en:", RUTA_LOCAL)

Descargado en: data/Madrid_Sale.rda


Celda 4. Registra la procedencia: URL, commit, fecha de descarga, tamaño y huella SHA-256. La huella sirve para comprobar que el archivo es el mismo si alguien lo repite.

In [5]:
contenido = RUTA_LOCAL.read_bytes()
fecha_descarga = datetime.fromtimestamp(RUTA_LOCAL.stat().st_mtime, tz=timezone.utc)

procedencia = {
    "url": URL,
    "repositorio": f"https://github.com/{REPO}",
    "commit": COMMIT,
    "ruta_en_repositorio": RUTA_EN_REPO,
    "licencia_declarada": "ODbL 1.0",
    "fecha_descarga_utc": fecha_descarga.strftime("%Y-%m-%d %H:%M:%S"),
    "tamano_bytes": len(contenido),
    "sha256": hashlib.sha256(contenido).hexdigest(),
}

for clave, valor in procedencia.items():
    print(f"{clave}: {valor}")

url: https://raw.githubusercontent.com/paezha/idealista18/e9ef80618f8a9cc28f672ac138f298bbd519918e/data/Madrid_Sale.rda
repositorio: https://github.com/paezha/idealista18
commit: e9ef80618f8a9cc28f672ac138f298bbd519918e
ruta_en_repositorio: data/Madrid_Sale.rda
licencia_declarada: ODbL 1.0
fecha_descarga_utc: 2026-11-16 14:53:22
tamano_bytes: 6738247
sha256: c50e281ee699a8a523a5505dd64eb1d5aa0e3ba712092355850ed6d9f0d38a20


Celda 5. Lee el archivo RDA. Un RDA es un contenedor de R que puede guardar varios objetos, y rdata lo devuelve como un diccionario de Python.

In [6]:
objeto_rda = rdata.read_rda(RUTA_LOCAL)

print(type(objeto_rda))
print(list(objeto_rda.keys()))

/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "sfg". The underlying R object is returned instead.
  warnings.warn(
/usr/local/lib/python3.

<class 'dict'>
['Madrid_Sale']


Celda 6. Saca la tabla del diccionario y comprueba sus dimensiones.

In [7]:
madrid = objeto_rda["Madrid_Sale"]

print(type(madrid))
print("Filas y columnas:", madrid.shape)
print("Primeros valores del índice:", list(madrid.index[:5]))

<class 'pandas.core.frame.DataFrame'>
Filas y columnas: (94815, 42)
Primeros valores del índice: [1, 2, 3, 4, 5]


Celda 7. Lista todas las columnas con su tipo de dato.

In [8]:
resumen_tipos = madrid.dtypes.astype(str).rename("tipo").to_frame()
resumen_tipos.index.name = "columna"
print(resumen_tipos.to_string())
print()
print(madrid.dtypes.astype(str).value_counts().to_string())

                                   tipo
columna                                
ASSETID                        category
PERIOD                            Int32
PRICE                           float64
UNITPRICE                       float64
CONSTRUCTEDAREA                   Int32
ROOMNUMBER                        Int32
BATHNUMBER                        Int32
HASTERRACE                        Int32
HASLIFT                           Int32
HASAIRCONDITIONING                Int32
AMENITYID                         Int32
HASPARKINGSPACE                   Int32
ISPARKINGSPACEINCLUDEDINPRICE     Int32
PARKINGSPACEPRICE                 Int32
HASNORTHORIENTATION               Int32
HASSOUTHORIENTATION               Int32
HASEASTORIENTATION                Int32
HASWESTORIENTATION                Int32
HASBOXROOM                        Int32
HASWARDROBE                       Int32
HASSWIMMINGPOOL                   Int32
HASDOORMAN                        Int32
HASGARDEN                         Int32


Celda 8. Muestra las 3 primeras filas girando la tabla (.T), para que las 42 columnas quepan en vertical.

In [9]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

madrid.head(3).T

,1,2,3
ASSETID,A15019136831406238029,A6677225905472065344,A13341979748618524775
PERIOD,201803,201803,201803
PRICE,126000.0,235000.0,373000.0
UNITPRICE,2680.851064,4351.851852,4973.333333
CONSTRUCTEDAREA,47,54,75
ROOMNUMBER,1,1,2
BATHNUMBER,1,1,1
HASTERRACE,0,0,0
HASLIFT,1,0,0
HASAIRCONDITIONING,1,0,1


Celda 9. Mira qué contiene la columna geometry.

In [10]:
print(type(madrid["geometry"].iloc[0]))
print(madrid["geometry"].iloc[0])

<class 'numpy.ndarray'>
[-3.76693283 40.36248477]


Celda 10. Comprueba si geometry coincide con LONGITUDE y LATITUDE en todas las filas, no solo en la primera.

In [11]:
coords_geom = np.vstack(madrid["geometry"].to_numpy())

dif_lon = np.abs(coords_geom[:, 0] - madrid["LONGITUDE"].to_numpy(dtype=float))
dif_lat = np.abs(coords_geom[:, 1] - madrid["LATITUDE"].to_numpy(dtype=float))

print("Forma de las coordenadas de geometry:", coords_geom.shape)
print("Máxima diferencia en longitud:", dif_lon.max())
print("Máxima diferencia en latitud:", dif_lat.max())
print("Filas con diferencia mayor que 1e-6:", int(((dif_lon > 1e-6) | (dif_lat > 1e-6)).sum()))

Forma de las coordenadas de geometry: (94815, 2)
Máxima diferencia en longitud: 0.0
Máxima diferencia en latitud: 0.0
Filas con diferencia mayor que 1e-6: 0


Celda 11. Cuenta los periodos y los inmuebles distintos de cada uno.

In [12]:
datos = madrid.drop(columns="geometry")

por_periodo = datos.groupby("PERIOD").agg(
    filas=("ASSETID", "size"),
    assetid_distintos=("ASSETID", "nunique"),
)
print(por_periodo.to_string())
print()
print("Periodos distintos:", sorted(datos["PERIOD"].dropna().unique().tolist()))
print("Periodos ausentes:", int(datos["PERIOD"].isna().sum()))

        filas  assetid_distintos
PERIOD                          
201803  21920              17622
201806  12652              12518
201809  15973              14246
201812  44270              31418

Periodos distintos: [201803, 201806, 201809, 201812]
Periodos ausentes: 0


Celda 12. Cuenta los nulos por columna, solo las que tienen alguno.

In [13]:
nulos = datos.isna().sum()
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": (nulos / len(datos) * 100).round(2),
})
print(resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False).to_string())
print()
print("Columnas con algún nulo:", int((nulos > 0).sum()), "de", datos.shape[1])

                    nulos  porcentaje
CONSTRUCTIONYEAR    55873       58.93
FLATLOCATIONID       6387        6.74
FLOORCLEAN           3846        4.06
CADASTRALQUALITYID      1        0.00

Columnas con algún nulo: 4 de 41


Celda 13. Distingue duplicados exactos de inmuebles repetidos.

In [14]:
print("Filas totales:", len(datos))
print("Duplicados exactos (fila completa):", int(datos.duplicated().sum()))
print("ASSETID distintos:", datos["ASSETID"].nunique())
print("Filas cuyo ASSETID aparece más de una vez:", int(datos["ASSETID"].duplicated(keep=False).sum()))
print("Filas en pares ASSETID-PERIOD repetidos:", int(datos.duplicated(subset=["ASSETID", "PERIOD"], keep=False).sum()))
print()
veces = datos["ASSETID"].value_counts()
print("Cuántos ASSETID aparecen 1, 2, 3... veces:")
print(veces.value_counts().sort_index().to_string())

Filas totales: 94815
Duplicados exactos (fila completa): 0
ASSETID distintos: 75804
Filas cuyo ASSETID aparece más de una vez: 32840
Filas en pares ASSETID-PERIOD repetidos: 32840

Cuántos ASSETID aparecen 1, 2, 3... veces:
count
0      1058
1     61975
2      9842
3      3098
4       673
5       150
6        51
7        11
8         1
9         2
11        1


Celda 14. Comprueba cuántos periodos distintos tiene cada ASSETID y cuántas filas hay como máximo por par ASSETID–PERIOD.

In [15]:
periodos_por_id = datos.groupby("ASSETID", observed=True)["PERIOD"].nunique()
print("Periodos distintos por ASSETID (cuántos ASSETID hay de cada caso):")
print(periodos_por_id.value_counts().sort_index().to_string())
print()

filas_por_par = datos.groupby(["ASSETID", "PERIOD"], observed=True).size()
print("Máximo de filas por par ASSETID-PERIOD:", int(filas_por_par.max()))

Periodos distintos por ASSETID (cuántos ASSETID hay de cada caso):
PERIOD
1    75804

Máximo de filas por par ASSETID-PERIOD: 11


Celda 15. Mira en qué columnas se diferencian las filas de un mismo ASSETID. Para cada columna cuenta en cuántos grupos de repetidos los valores no son todos iguales.

In [16]:
repetidos = datos[datos["ASSETID"].duplicated(keep=False)].copy()
repetidos["ASSETID"] = repetidos["ASSETID"].astype(str)

columnas = [c for c in repetidos.columns if c != "ASSETID"]
difiere = (
    repetidos.groupby("ASSETID")[columnas]
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

print("Grupos de ASSETID repetidos:", repetidos["ASSETID"].nunique())
print()
print("En cuántos grupos difiere cada columna:")
print(difiere.to_string())

Grupos de ASSETID repetidos: 13829

En cuántos grupos difiere cada columna:
DISTANCE_TO_CITY_CENTER          13825
DISTANCE_TO_CASTELLANA           13825
LONGITUDE                        13825
DISTANCE_TO_METRO                13825
LATITUDE                         13825
UNITPRICE                        13548
PRICE                            13539
CONSTRUCTEDAREA                    974
CONSTRUCTIONYEAR                   466
AMENITYID                          364
ISPARKINGSPACEINCLUDEDINPRICE      363
HASPARKINGSPACE                    363
FLOORCLEAN                         282
HASTERRACE                         276
ROOMNUMBER                         253
FLATLOCATIONID                     237
HASSOUTHORIENTATION                236
HASDOORMAN                         231
HASEASTORIENTATION                 226
HASWARDROBE                        205
HASAIRCONDITIONING                 202
HASWESTORIENTATION                 200
PARKINGSPACEPRICE                  188
BUILTTYPEID_3              

Celda 16. Enseña un ejemplo concreto, para ver con tus ojos qué cambia.

In [17]:
ejemplo_id = repetidos["ASSETID"].iloc[0]
print("ASSETID del ejemplo:", ejemplo_id)
repetidos[repetidos["ASSETID"] == ejemplo_id].T

ASSETID del ejemplo: A4775182175615276542


,4,3344,78833
ASSETID,A4775182175615276542,A4775182175615276542,A4775182175615276542
PERIOD,201803,201803,201803
PRICE,284000.0,258000.0,257000.0
UNITPRICE,5916.666667,5733.333333,6425.0
CONSTRUCTEDAREA,48,45,40
ROOMNUMBER,1,1,1
BATHNUMBER,1,1,1
HASTERRACE,0,0,0
HASLIFT,1,1,1
HASAIRCONDITIONING,1,1,1


Celda 17. Resumen de las variables numéricas, con percentiles para ver los extremos sin quitarlos.

In [18]:
numericas = [
    "PRICE", "UNITPRICE", "CONSTRUCTEDAREA", "ROOMNUMBER", "BATHNUMBER",
    "CONSTRUCTIONYEAR", "CADCONSTRUCTIONYEAR", "CADMAXBUILDINGFLOOR", "CADDWELLINGCOUNT",
    "DISTANCE_TO_CITY_CENTER", "DISTANCE_TO_METRO", "DISTANCE_TO_CASTELLANA",
    "PARKINGSPACEPRICE", "LONGITUDE", "LATITUDE",
]

resumen = datos[numericas].astype("float64").describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
print(resumen.round(2).to_string())

                           count       mean        std       min        1%        5%        25%        50%        75%         95%         99%         max
PRICE                    94815.0  396110.11  417074.41  21000.00  71000.00  97000.00  160000.00  262000.00  467000.00  1135000.00  2207000.00  8133000.00
UNITPRICE                94815.0    3661.05    1700.50    805.31   1088.89   1426.67    2240.00    3480.00    4744.62     6771.43     8584.21     9997.56
CONSTRUCTEDAREA          94815.0     101.40      67.08     21.00     30.00     40.00      62.00      83.00     117.00      225.00      367.00      985.00
ROOMNUMBER               94815.0       2.58       1.24      0.00      0.00      1.00       2.00       3.00       3.00        4.00        6.00       93.00
BATHNUMBER               94815.0       1.59       0.84      0.00      1.00      1.00       1.00       1.00       2.00        3.00        5.00       20.00
CONSTRUCTIONYEAR         38942.0    1964.69      55.89      1.00   1880.00  

Celda 18. Variables de tipo 0/1: cuántos ceros, unos, otros valores y nulos hay en cada una.

In [19]:
binarias = [c for c in datos.columns if c.startswith(("HAS", "IS", "BUILTTYPEID_"))]

filas = []
for c in binarias:
    s = datos[c].astype("float64")
    filas.append({
        "columna": c,
        "valor_0": int((s == 0).sum()),
        "valor_1": int((s == 1).sum()),
        "otros": int((s.notna() & ~s.isin([0, 1])).sum()),
        "nulos": int(s.isna().sum()),
    })

print(pd.DataFrame(filas).set_index("columna").to_string())

                               valor_0  valor_1  otros  nulos
columna                                                      
HASTERRACE                       61131    33684      0      0
HASLIFT                          28862    65953      0      0
HASAIRCONDITIONING               52281    42534      0      0
HASPARKINGSPACE                  73472    21343      0      0
ISPARKINGSPACEINCLUDEDINPRICE    73472    21343      0      0
HASNORTHORIENTATION              84592    10223      0      0
HASSOUTHORIENTATION              72427    22388      0      0
HASEASTORIENTATION               75602    19213      0      0
HASWESTORIENTATION               80739    14076      0      0
HASBOXROOM                       70272    24543      0      0
HASWARDROBE                      40580    54235      0      0
HASSWIMMINGPOOL                  80767    14048      0      0
HASDOORMAN                       71151    23664      0      0
HASGARDEN                        77319    17496      0      0
ISDUPLEX

Celda 19. Variables con códigos de categoría.

In [20]:
codigos = ["AMENITYID", "FLOORCLEAN", "FLATLOCATIONID", "CADASTRALQUALITYID"]

for c in codigos:
    print(c)
    print(datos[c].value_counts(dropna=False).sort_index().to_string())
    print()

AMENITYID
AMENITYID
1     1317
2     4699
3    88799

FLOORCLEAN
FLOORCLEAN
-1        936
0       10112
1       20336
2       17044
3       14754
4       11635
5        6162
6        3733
7        2447
8        1495
9         814
10        450
11       1051
<NA>     3846

FLATLOCATIONID
FLATLOCATIONID
1       76324
2       12104
<NA>     6387

CADASTRALQUALITYID
CADASTRALQUALITYID
0         377
1         629
2        2703
3       12634
4       24673
5       20725
6       20491
7       10428
8        1527
9         627
<NA>        1



Celda 20. Cuántas filas están lejos del centro y cómo son.

In [21]:
col = "DISTANCE_TO_CITY_CENTER"

for umbral in [12, 15, 20, 30, 50, 100]:
    print(f"Filas con {col} > {umbral} km:", int((datos[col] > umbral).sum()))

print()
lejanas = datos[datos[col] > 20].sort_values(col, ascending=False)
ver = ["ASSETID", "PERIOD", "LONGITUDE", "LATITUDE", col, "DISTANCE_TO_METRO", "UNITPRICE", "CONSTRUCTEDAREA"]
print(lejanas[ver].head(15).to_string())

Filas con DISTANCE_TO_CITY_CENTER > 12 km: 723
Filas con DISTANCE_TO_CITY_CENTER > 15 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 20 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 30 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 50 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 100 km: 1

                    ASSETID  PERIOD  LONGITUDE   LATITUDE  DISTANCE_TO_CITY_CENTER  DISTANCE_TO_METRO    UNITPRICE  CONSTRUCTEDAREA
84001  A7501251371917084834  201806  -2.753303  36.756391               415.752584         399.477366  1317.241379              145


Celda 21. Habitaciones y baños.

In [22]:
hab = datos["ROOMNUMBER"]

print("ROOMNUMBER, valores de 0 a 10:")
print(hab[hab <= 10].value_counts().sort_index().to_string())
print("Filas con 11 o más habitaciones:", int((hab >= 11).sum()))
print()
print(datos[hab >= 11][["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE", "UNITPRICE"]]
      .sort_values("ROOMNUMBER", ascending=False).head(10).to_string())
print()
print("Filas con ROOMNUMBER = 0, según ISSTUDIO:")
print(datos.loc[hab == 0, "ISSTUDIO"].value_counts().to_string())
print()
print("BATHNUMBER:")
print(datos["BATHNUMBER"].value_counts().sort_index().to_string())

ROOMNUMBER, valores de 0 a 10:
ROOMNUMBER
0      2745
1     13338
2     28422
3     33961
4     11674
5      3350
6       789
7       279
8       142
9        35
10       24
Filas con 11 o más habitaciones: 56

       ROOMNUMBER  BATHNUMBER  CONSTRUCTEDAREA      PRICE    UNITPRICE
17639          93           2              119   450000.0  3781.512605
84060          33           2               90   109000.0  1211.111111
68892          20          20              600   850000.0  1416.666667
545            20          16              847  2020000.0  2384.887839
25849          18           4              765  6970000.0  9111.111111
60434          18          18              455  1469000.0  3228.571429
56804          16          14              680  3375000.0  4963.235294
63967          16          10              390  1956000.0  5015.384615
11072          15          11              489  1823000.0  3728.016360
19018          15          11              489  1890000.0  3865.030675

Filas c

Celda 22. Aparcamiento: comprobar las tres columnas que parecen relacionadas.

In [23]:
print(pd.crosstab(datos["HASPARKINGSPACE"], datos["ISPARKINGSPACEINCLUDEDINPRICE"],
                  rownames=["HASPARKINGSPACE"], colnames=["INCLUIDO_EN_PRECIO"]).to_string())
print()
print("¿HASPARKINGSPACE e ISPARKINGSPACEINCLUDEDINPRICE son idénticas fila a fila?:",
      bool((datos["HASPARKINGSPACE"] == datos["ISPARKINGSPACEINCLUDEDINPRICE"]).all()))
print()

es_uno = datos["PARKINGSPACEPRICE"] == 1
print(pd.crosstab(datos["HASPARKINGSPACE"], es_uno,
                  rownames=["HASPARKINGSPACE"], colnames=["PARKINGSPACEPRICE == 1"]).to_string())
print()
print("PARKINGSPACEPRICE cuando es distinto de 1:")
print(datos.loc[~es_uno, "PARKINGSPACEPRICE"].describe().round(0).to_string())

INCLUIDO_EN_PRECIO      0      1
HASPARKINGSPACE                 
0                   73472      0
1                       0  21343

¿HASPARKINGSPACE e ISPARKINGSPACEINCLUDEDINPRICE son idénticas fila a fila?: True

PARKINGSPACEPRICE == 1  False  True 
HASPARKINGSPACE                     
0                        2183  71289
1                           8  21335

PARKINGSPACEPRICE cuando es distinto de 1:
count      2191.0
mean      31110.0
std       38710.0
min           2.0
25%       16001.0
50%       25001.0
75%       40001.0
max      925001.0


Celda 23. Mira los años imposibles y compara los dos años de construcción donde ambos existen.

In [24]:
cy = datos["CONSTRUCTIONYEAR"].astype("float64")
cc = datos["CADCONSTRUCTIONYEAR"].astype("float64")

print("CONSTRUCTIONYEAR con valor:", int(cy.notna().sum()))
print("  menor que 1800:", int((cy < 1800).sum()), "| mayor que 2018:", int((cy > 2018).sum()))
print("CADCONSTRUCTIONYEAR menor que 1800:", int((cc < 1800).sum()))
print()
print("Valores extremos de CONSTRUCTIONYEAR (hasta 30):")
print(cy[(cy < 1800) | (cy > 2018)].value_counts().sort_index().head(30).to_string())
print()
ambos = cy.notna()
dif = cy[ambos] - cc[ambos]
print("Filas con ambos años:", int(ambos.sum()))
print("Años exactamente iguales:", int((dif == 0).sum()))
print("CONSTRUCTIONYEAR - CADCONSTRUCTIONYEAR:")
print(dif.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1).to_string())

CONSTRUCTIONYEAR con valor: 38942
  menor que 1800: 41 | mayor que 2018: 129
CADCONSTRUCTIONYEAR menor que 1800: 13

Valores extremos de CONSTRUCTIONYEAR (hasta 30):
CONSTRUCTIONYEAR
1.0         1
7.0         1
10.0        2
19.0        3
48.0        1
49.0        2
50.0        2
54.0        1
160.0       1
173.0       1
190.0       1
195.0       1
198.0       4
225.0       1
350.0       1
1056.0      1
1067.0      1
1070.0      1
1073.0      1
1197.0      1
1623.0      2
1627.0      3
1655.0      1
1692.0      1
1696.0      1
1700.0      1
1723.0      1
1730.0      2
1780.0      1
2019.0    114

Filas con ambos años: 38942
Años exactamente iguales: 38464
CONSTRUCTIONYEAR - CADCONSTRUCTIONYEAR:
count    38942.0
mean        -1.1
std         46.6
min      -2001.0
5%           0.0
25%          0.0
50%          0.0
75%          0.0
95%          0.0
max        327.0


Celda 24. Cola alta de UNITPRICE y comprobación de que UNITPRICE = PRICE / CONSTRUCTEDAREA en todas las filas (antes solo lo vimos en 3).

In [25]:
u = datos["UNITPRICE"]
for t in [9000, 9500, 9900, 9990]:
    print(f"UNITPRICE mayor que {t}:", int((u > t).sum()))
print()
print("Los 10 valores más altos:")
print(u.sort_values(ascending=False).head(10).round(2).to_string())
print()
calc = datos["PRICE"].astype("float64") / datos["CONSTRUCTEDAREA"].astype("float64")
dif_u = (calc - u).abs()
print("Máxima diferencia |PRICE/CONSTRUCTEDAREA - UNITPRICE|:", float(dif_u.max()))
print("Filas con diferencia mayor que 0.01:", int((dif_u > 0.01).sum()))

UNITPRICE mayor que 9000: 545
UNITPRICE mayor que 9500: 214
UNITPRICE mayor que 9900: 46
UNITPRICE mayor que 9990: 6

Los 10 valores más altos:
79382    9997.56
79287    9994.29
46887    9993.38
2600     9992.25
36579    9991.15
56953    9990.91
2590     9984.92
92912    9979.31
35298    9975.00
11526    9975.00

Máxima diferencia |PRICE/CONSTRUCTEDAREA - UNITPRICE|: 6.366462912410498e-12
Filas con diferencia mayor que 0.01: 0


Celda 25. ¿Son los tres BUILTTYPEID_* excluyentes?

In [26]:
bt = datos[["BUILTTYPEID_1", "BUILTTYPEID_2", "BUILTTYPEID_3"]].astype("int64")
print("Cuántos BUILTTYPEID_* valen 1 en cada fila:")
print(bt.sum(axis=1).value_counts().sort_index().to_string())

Cuántos BUILTTYPEID_* valen 1 en cada fila:
1    94815


Celda 26. Cómo se reparte UNITPRICE por periodo.

In [27]:
por_periodo = datos.groupby("PERIOD").agg(
    filas=("UNITPRICE", "size"),
    mediana=("UNITPRICE", "median"),
    media=("UNITPRICE", "mean"),
    p95=("UNITPRICE", lambda s: s.quantile(0.95)),
)
print(por_periodo.round(0).to_string())

        filas  mediana   media     p95
PERIOD                                
201803  21920   3314.0  3520.0  6639.0
201806  12652   3376.0  3569.0  6542.0
201809  15973   3463.0  3647.0  6769.0
201812  44270   3585.0  3762.0  6877.0


Celda 27. Ascensor y precio por planta (te sirve para H3 y para entender los códigos −1 y 11 de FLOORCLEAN).

In [28]:
g = datos.groupby("FLOORCLEAN", observed=True, dropna=False).agg(
    filas=("HASLIFT", "size"),
    pct_ascensor=("HASLIFT", "mean"),
    unitprice_mediana=("UNITPRICE", "median"),
)
g["pct_ascensor"] = (g["pct_ascensor"] * 100).round(1)
g["unitprice_mediana"] = g["unitprice_mediana"].round(0)
print(g.to_string())

            filas  pct_ascensor  unitprice_mediana
FLOORCLEAN                                        
-1            936          60.3             3742.0
0           10112          52.7             2857.0
1           20336          67.8             3351.0
2           17044          64.5             3471.0
3           14754          63.7             3428.0
4           11635          69.9             3622.0
5            6162          88.4             4057.0
6            3733          99.2             4227.0
7            2447          99.9             4000.0
8            1495          99.9             3955.0
9             814         100.0             3502.0
10            450         100.0             3585.0
11           1051          99.5             3831.0
<NA>         3846          61.3             3155.0


Celda 28. Cuenta las filas con y sin ascensor por tramo de planta, con la mediana de precio de cada grupo.

In [29]:
d = datos[datos["FLOORCLEAN"].notna()].copy()
d["tramo_planta"] = pd.cut(
    d["FLOORCLEAN"].astype("float64"),
    bins=[-2, 0, 2, 4, 6, 20],
    labels=["-1 a 0", "1 a 2", "3 a 4", "5 a 6", "7 o más"],
)

t = d.groupby(["tramo_planta", "HASLIFT"], observed=True).agg(
    filas=("UNITPRICE", "size"),
    mediana_unitprice=("UNITPRICE", "median"),
)
t["mediana_unitprice"] = t["mediana_unitprice"].round(0)
print(t.to_string())

                      filas  mediana_unitprice
tramo_planta HASLIFT                          
-1 a 0       0         5155             2380.0
             1         5893             3401.0
1 a 2        0        12609             2262.0
             1        24771             3843.0
3 a 4        0         8860             2161.0
             1        17529             4025.0
5 a 6        0          742             2088.0
             1         9153             4215.0
7 o más      0            9             3050.0
             1         6248             3858.0


Celda 29. Primero, cuántos inmuebles distintos hay en el grupo de 742 filas, y después una tabla de «marcas».

In [30]:
# 1) El grupo sin ascensor en plantas 5 a 6: filas frente a inmuebles distintos
sin_asc = d[(d["tramo_planta"] == "5 a 6") & (d["HASLIFT"] == 0)]
print("Plantas 5 a 6 sin ascensor -> filas:", len(sin_asc),
      "| ASSETID distintos:", sin_asc["ASSETID"].nunique())
print()

# 2) Marcas de posibles problemas (solo se cuentan, no se elimina nada)
anio = datos["CONSTRUCTIONYEAR"].astype("float64")
marcas = pd.DataFrame({
    "lejos_de_madrid_mas_15km": datos["DISTANCE_TO_CITY_CENTER"] > 15,
    "habitaciones_11_o_mas": datos["ROOMNUMBER"] >= 11,
    "habitaciones_0_no_estudio": (datos["ROOMNUMBER"] == 0) & (datos["ISSTUDIO"] == 0),
    "banos_0": datos["BATHNUMBER"] == 0,
    "anio_antes_1800": anio < 1800,
    "anio_despues_2018": anio > 2018,
})
print(marcas.sum().to_string())
print()
print("Filas con al menos una marca:", int(marcas.any(axis=1).sum()))
print("Filas con dos o más marcas:", int((marcas.sum(axis=1) >= 2).sum()))

Plantas 5 a 6 sin ascensor -> filas: 742 | ASSETID distintos: 593

lejos_de_madrid_mas_15km       1
habitaciones_11_o_mas         56
habitaciones_0_no_estudio    161
banos_0                       89
anio_antes_1800               41
anio_despues_2018            129

Filas con al menos una marca: 430
Filas con dos o más marcas: 47


Bloque 11: un criterio objetivo para las habitaciones

Para decidir qué filas de habitaciones son imposibles, un número ayuda más que mirarlas una a una: los metros cuadrados por habitación.

Celda 30.

In [31]:
r = datos[datos["ROOMNUMBER"] > 0].copy()
r["m2_por_hab"] = r["CONSTRUCTEDAREA"].astype("float64") / r["ROOMNUMBER"].astype("float64")

print(r["m2_por_hab"].describe(percentiles=[0.001, 0.01, 0.05, 0.5, 0.95]).round(1).to_string())
print()
for u in [5, 8, 10, 12]:
    print(f"Filas con menos de {u} m² por habitación:", int((r["m2_por_hab"] < u).sum()))
print()
cols = ["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE", "ISSTUDIO"]
print(r.sort_values("m2_por_hab").head(10)[cols].to_string())

count    92070.0
mean        39.9
std         19.7
min          1.3
0.1%        14.0
1%          17.7
5%          21.3
50%         35.5
95%         72.0
max        928.0

Filas con menos de 5 m² por habitación: 2
Filas con menos de 8 m² por habitación: 2
Filas con menos de 10 m² por habitación: 9
Filas con menos de 12 m² por habitación: 22

       ROOMNUMBER  BATHNUMBER  CONSTRUCTEDAREA     PRICE  ISSTUDIO
17639          93           2              119  450000.0         0
84060          33           2               90  109000.0         0
1646            3           0               25  119000.0         0
7525            3           1               25   54000.0         0
75515           6           1               55  188000.0         0
49816           3           1               28   59000.0         0
14135           3           1               28   80000.0         0
33411           3           1               28  103000.0         0
14165           3           1               28   69000

Celda 31. Usa la variable r

In [32]:
for u in [100, 150, 300]:
    print(f"Filas con más de {u} m² por habitación:", int((r["m2_por_hab"] > u).sum()))
print()

alt = r[r["m2_por_hab"] > 150].sort_values("m2_por_hab", ascending=False)
cols = ["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE", "UNITPRICE", "ISSTUDIO"]
print(alt.head(10)[cols].to_string())

Filas con más de 100 m² por habitación: 839
Filas con más de 150 m² por habitación: 130
Filas con más de 300 m² por habitación: 33

       ROOMNUMBER  BATHNUMBER  CONSTRUCTEDAREA      PRICE    UNITPRICE  ISSTUDIO
42805           1           1              928  2732000.0  2943.965517         0
63226           1           1              926  4098000.0  4425.485961         0
63544           1           1              922  3060000.0  3318.872017         0
74176           1           1              800   747000.0   933.750000         0
34925           1           1              732  4925000.0  6728.142077         0
75578           1           1              724  2307000.0  3186.464088         0
81665           1           1              712  1888000.0  2651.685393         0
78518           1           1              650   652000.0  1003.076923         0
75330           1           1              613  2065000.0  3368.678630         0
23378           1           1              600   867000.0 

Bloque 13: ¿es raro tener mucha superficie con pocas habitaciones?

Para encontrar un criterio mejor, miramos la superficie típica de cada tamaño de vivienda y cuántas filas pasan de 300 m² según su número de habitaciones.

Celda 32.

In [33]:
g = datos[datos["ROOMNUMBER"].between(1, 6)].groupby("ROOMNUMBER")["CONSTRUCTEDAREA"]
print("Superficie (m²) según número de habitaciones:")
print(g.describe(percentiles=[0.5, 0.95, 0.99]).round(0).to_string())
print()

grandes = datos[datos["CONSTRUCTEDAREA"] > 300]
print("Filas con más de 300 m², según ROOMNUMBER (8 = 8 o más):")
print(grandes["ROOMNUMBER"].clip(upper=8).value_counts().sort_index().to_string())

Superficie (m²) según número de habitaciones:
              count   mean    std   min    50%    95%    99%    max
ROOMNUMBER                                                         
1           13338.0   56.0   32.0  21.0   51.0   91.0  135.0  928.0
2           28422.0   76.0   28.0  21.0   70.0  126.0  173.0  950.0
3           33961.0  102.0   42.0  25.0   93.0  177.0  250.0  985.0
4           11674.0  161.0   67.0  52.0  144.0  299.0  385.0  951.0
5            3350.0  236.0  100.0  78.0  212.0  430.0  572.0  941.0
6             789.0  296.0  140.0  55.0  260.0  540.0  805.0  982.0

Filas con más de 300 m², según ROOMNUMBER (8 = 8 o más):
ROOMNUMBER
0     24
1     31
2      9
3    117
4    507
5    678
6    319
7    122
8    136


celda 33

In [34]:
datos_rev = datos.copy()
area = datos_rev["CONSTRUCTEDAREA"].astype("float64")
hab = datos_rev["ROOMNUMBER"].astype("float64")

datos_rev["flag_fuera_madrid"] = datos_rev["DISTANCE_TO_CITY_CENTER"] > 50
datos_rev["flag_hab_imposible"] = (hab > 0) & ((area / hab) < 5)
datos_rev["flag_sup_grande_pocas_hab"] = (area > 300) & (hab <= 2)
datos_rev["flag_banos_0"] = datos_rev["BATHNUMBER"] == 0

flags = [c for c in datos_rev.columns if c.startswith("flag_")]
print(datos_rev[flags].sum().to_string())

datos_rev["alguna_flag"] = datos_rev[flags].any(axis=1)
print()
print("Filas con alguna marca:", int(datos_rev["alguna_flag"].sum()))
print()
print("UNITPRICE según tenga o no alguna marca:")
print(datos_rev.groupby("alguna_flag")["UNITPRICE"].agg(["size", "mean", "median"]).round(1).to_string())

flag_fuera_madrid             1
flag_hab_imposible            2
flag_sup_grande_pocas_hab    64
flag_banos_0                 89

Filas con alguna marca: 152

UNITPRICE según tenga o no alguna marca:
              size    mean  median
alguna_flag                       
False        94663  3661.6  3480.0
True           152  3323.1  3170.2
